# 15 · Tests

Checks of `buffalo_data.py`, `buffalo_validation.py` and `sightings_map.py`: calculations on small hand-made inputs with known answers, then sanity checks of the camera data and the cached real data. No Earth Engine.

In [2]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from pyproj import Geod
from shapely.geometry import box

import buffalo_data as bd
import buffalo_validation as bv
import sightings_map as sm
from buffalo_data import BAND_COLS, COVARIATE_COLS, BuffaloConfig

config = BuffaloConfig(cache_dir="cache")
results = []


def check(name, condition, detail=""):
    results.append({"test": name, "result": "PASS" if bool(condition) else "FAIL", "detail": detail})


def skip(name, reason):
    results.append({"test": name, "result": "SKIP", "detail": reason})


def utm_points(xy, years, ids=None):
    lon, lat = bd.to_lonlat.transform(np.array(xy)[:, 0], np.array(xy)[:, 1])
    df = pd.DataFrame({"decimalLongitude": lon, "decimalLatitude": lat, "year": np.array(years, dtype=float)})
    if ids is not None:
        df.insert(0, "gbifID", ids)
    return df


SERONERA = bd.to_utm.transform(34.83, -2.45)

## 1. Coordinates and settings

In [3]:
lons, lats = np.array([34.0, 34.83, 36.0]), np.array([-3.6, -2.45, -1.6])
x, y = bd.to_utm.transform(lons, lats)
lon2, lat2 = bd.to_lonlat.transform(x, y)
check("UTM round trip", np.abs(lon2 - lons).max() < 1e-7 and np.abs(lat2 - lats).max() < 1e-7)

x1, y1 = bd.to_utm.transform(34.83, -2.45)
x2, y2 = bd.to_utm.transform(34.93, -2.55)
utm_distance = np.hypot(x2 - x1, y2 - y1)
geodesic = Geod(ellps="WGS84").inv(34.83, -2.45, 34.93, -2.55)[2]
check("UTM distance matches geodesic within 0.1 %", abs(utm_distance / geodesic - 1) < 0.001,
      f"{utm_distance:.1f} m vs {geodesic:.1f} m")

check("config key ignores cache_dir", BuffaloConfig(cache_dir="a").key() == BuffaloConfig(cache_dir=None).key())
check("config key changes with buffer", BuffaloConfig(buffer_m=500).key() != BuffaloConfig().key())

## 2. Thinning and visited cells

In [4]:
cx, cy = (SERONERA[0] // 1000) * 1000, (SERONERA[1] // 1000) * 1000
pts = utm_points([(cx + 100, cy + 100), (cx + 900, cy + 900), (cx + 500, cy + 500), (cx + 1500, cy + 500)],
                 [2020, 2020, 2021, 2020], ids=["a", "b", "c", "d"])
thinned = bd.thin_presences(pts, config)
check("thinning keeps one per 1 km cell per year", sorted(thinned["gbifID"]) == ["a", "c", "d"], str(sorted(thinned["gbifID"])))

cells = bd.find_visited_cells(utm_points([(cx + 100, cy + 100), (cx + 25_000, cy + 100)], [2020, 2020]), config)
expected = {(int((cx + 100) // 10_000), int((cy + 100) // 10_000)), (int((cx + 25_000) // 10_000), int((cy + 100) // 10_000))}
check("visited cells from mammal records", cells == expected, str(cells))

## 3. Background sampling

In [5]:
rng = np.random.default_rng(1)
base_x, base_y = SERONERA
presence_xy = np.c_[base_x + rng.uniform(-20_000, 20_000, 30), base_y + rng.uniform(-20_000, 20_000, 30)]
years = rng.choice([2018, 2020, 2022], 30)
presences = utm_points(presence_xy, years, ids=[f"p{i}" for i in range(30)])
visited = {(int((base_x + dx) // 10_000), int((base_y + dy) // 10_000)) for dx in range(-30_000, 30_001, 10_000) for dy in range(-30_000, 30_001, 10_000)}

points = bd.make_presence_background(presences, config.bounds, visited, config)
bg = points[points["presence"] == 0]
check("background = n_per_presence x presences", len(bg) == config.n_per_presence * len(presences), f"{len(bg)}")
per_year = bg.groupby("year").size() / presences.groupby("year").size()
check("background matches presence years exactly", (per_year == config.n_per_presence).all(), per_year.to_dict())

bx, by = bd.to_utm.transform(bg["decimalLongitude"].values, bg["decimalLatitude"].values)
px, py = bd.to_utm.transform(presences["decimalLongitude"].values, presences["decimalLatitude"].values)
min_dist = min(np.hypot(px[presences["year"].values == yr] - xb, py[presences["year"].values == yr] - yb).min()
               for xb, yb, yr in zip(bx, by, bg["year"].values))
check("background >= buffer from same-year presences", min_dist >= config.buffer_m, f"closest {min_dist:.0f} m")
check("background inside visited cells", all((int(a // 10_000), int(b // 10_000)) in visited for a, b in zip(bx, by)))
west, south, east, north = config.bounds
check("background inside the box", bg["decimalLongitude"].between(west, east).all() and bg["decimalLatitude"].between(south, north).all())
again = bd.make_presence_background(presences, config.bounds, visited, config)
check("background reproducible with the seed", points.equals(again))
check("presence ids kept", set(points.loc[points["presence"] == 1, "point_id"]) == set(presences["gbifID"]))

## 4. Year matching and species background

In [6]:
df = pd.DataFrame({"year": [2017.0] * 2 + [2018.0] * 3 + [2017.0] * 30 + [2018.0] * 12,
                   "presence": [1] * 5 + [0] * 42})
matched, ratio = bv.year_matched(df)
counts = matched.groupby(["year", "presence"]).size()
check("year_matched picks the largest ratio every year supports", ratio == 4, f"ratio {ratio}")
check("year_matched draws ratio x presences per year",
      counts[(2017.0, 0)] == 8 and counts[(2018.0, 0)] == 12 and counts[(2017.0, 1)] == 2 and counts[(2018.0, 1)] == 3)

p1 = (base_x, base_y)
sp_presences = utm_points([p1, (base_x + 40_000, base_y)], [2017, 2018]).assign(point_id=["P1", "P2"])
pool_xy = [(base_x + 300, base_y), (base_x + 2_000, base_y), (base_x + 3_000, base_y), (base_x + 300, base_y), (base_x + 5_000, base_y)]
pool = utm_points(pool_xy, [2017, 2017, 2017, 2018, 2018]).assign(point_id=["a", "b", "c", "d", "e"])
species_df, ratio = bv.species_dataset(sp_presences, pool, config)
kept = sorted(species_df.loc[species_df["presence"] == 0, "point_id"])
check("species background drops pool points within buffer of a same-year presence", "a" not in kept and kept == ["b", "c", "d", "e"], str(kept))

## 5. Metrics

In [7]:
y_true = np.array([1, 1, 0, 0])
check("TSS = 0 for half right", abs(bv.tss(y_true, np.array([0.9, 0.2, 0.1, 0.8]), 0.5)) < 1e-12)
check("TSS = 1 for perfect", bv.tss(y_true, np.array([0.9, 0.8, 0.1, 0.2]), 0.5) == 1)
threshold = bv.best_tss_threshold(np.array([0, 0, 1, 1]), np.array([0.1, 0.2, 0.8, 0.9]))
check("best TSS threshold separates perfectly", 0.2 < threshold <= 0.8, f"{threshold}")

rng = np.random.default_rng(0)
p = rng.uniform(0.02, 0.98, 20_000)
y = rng.uniform(size=p.size) < p
logit = np.log(p / (1 - p))
check("calibration slope ~1 when calibrated", 0.9 < bv.calibration_slope(y, p) < 1.1, f"{bv.calibration_slope(y, p):.2f}")
over = 1 / (1 + np.exp(-2 * logit))
check("calibration slope ~0.5 when over-confident", 0.4 < bv.calibration_slope(y, over) < 0.6, f"{bv.calibration_slope(y, over):.2f}")

cams = pd.DataFrame({"rate": np.arange(30, dtype=float)})
agreement = bv.camera_agreement(cams, np.arange(30) / 30)
check("camera agreement rho = 1 for a perfect ranking", abs(agreement["camera rho"] - 1) < 1e-12)
check("camera agreement medians rise", agreement["median rate low"] < agreement["median rate medium"] < agreement["median rate high"])

## 6. Regions and protected areas

In [8]:
rng = np.random.default_rng(2)
cluster = lambda cx, cy: np.c_[rng.normal(cx, 500, 50), rng.normal(cy, 500, 50)]
xy = np.vstack([cluster(0, 0), cluster(50_000, 0)])
df = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "presence": 1})
centres = bv.presence_regions(df, 2)
check("k-means centres find the two clusters", sorted(np.round(centres[:, 0] / 10_000)) == [0, 5], str(np.round(centres)))
regions = bv.nearest_region(df, centres)
check("each cluster gets one region", len(set(regions[:50])) == 1 and len(set(regions[50:])) == 1 and regions[0] != regions[-1])

areas = gpd.GeoDataFrame({"area": ["Serengeti NP", "Ngorongoro CA"]},
                         geometry=[box(0, 0, 10_000, 10_000), box(5_000, 0, 20_000, 10_000)], crs=bd.UTM)
pts = pd.DataFrame({"x": [1_000, 7_000, 15_000, 30_000], "y": [5_000] * 4})
found = bv.protected_area_of(pts, areas).tolist()
check("protected area lookup, first in order wins on overlap",
      found == ["Serengeti NP", "Serengeti NP", "Ngorongoro CA", bv.OUTSIDE], str(found))

## 7. Blocked cross-validation

In [9]:
rng = np.random.default_rng(3)
n = 600
df = pd.DataFrame({"f1": rng.normal(size=n), "f2": rng.normal(size=n), "cell": rng.integers(0, 30, n).astype(str)})
df["presence"] = (df["f1"] + rng.normal(scale=0.3, size=n) > 1).astype(int)
groups = df["cell"].astype(int) % 3
splits = [(np.where(groups != g)[0], np.where(groups == g)[0]) for g in range(3)]
table, held_out, models = bv.blocked_cv(df, splits, ["a", "b", "c"], bv.XGB_SETTINGS["regularized"], ["f1", "f2"])
check("blocked CV: train and test never overlap", all(len(np.intersect1d(tr, te)) == 0 for tr, te in splits))
check("blocked CV: every point gets a held-out score", held_out.notna().all())
check("blocked CV: informative feature gives high AUC", table.loc["mean", "AUC"] > 0.85, f"AUC {table.loc['mean', 'AUC']}")
check("blocked CV: table has a mean row", "mean" in table.index)

wrapped = bv.FixedAtPrediction(models["a"], {"f1": 0.0})
varied = df[["f1", "f2"]].assign(f1=np.linspace(-3, 3, n))
check("fixed at prediction: the fixed column has no effect",
      np.allclose(wrapped.predict_proba(varied), wrapped.predict_proba(varied.assign(f1=5.0))))

## 8. Hotspot areas

In [10]:
spacing = 2_500
gx, gy = np.meshgrid(np.arange(20) * spacing + base_x, np.arange(20) * spacing + base_y)
grid = pd.DataFrame({"x": gx.ravel(), "y": gy.ravel()})
grid["decimalLongitude"], grid["decimalLatitude"] = bd.to_lonlat.transform(grid["x"].values, grid["y"].values)
peak1 = (base_x + 5 * spacing, base_y + 5 * spacing)
peak2 = (base_x + 15 * spacing, base_y + 14 * spacing)
score = (np.exp(-np.hypot(grid["x"] - peak1[0], grid["y"] - peak1[1]) ** 2 / (2 * 6_000 ** 2))
         + 0.8 * np.exp(-np.hypot(grid["x"] - peak2[0], grid["y"] - peak2[1]) ** 2 / (2 * 6_000 ** 2)))
cams = pd.DataFrame({"x": [peak1[0], peak2[0], base_x], "y": [peak1[1], peak2[1], base_y], "rate": [5.0, 2.0, 0.0]})
hot = bv.hotspot_areas(grid, score, cams, spacing, n_areas=3)

px, py = bd.to_utm.transform(hot.loc[0, "peak_lon"], hot.loc[0, "peak_lat"])
check("hotspot 1 is at the strongest peak", np.hypot(px - peak1[0], py - peak1[1]) <= spacing, f"{np.hypot(px - peak1[0], py - peak1[1]):.0f} m away")
check("at most n_areas hotspots", len(hot) <= 3)
utm_hot = hot.to_crs(bd.UTM)
check("hotspots don't overlap", all(utm_hot.geometry[i].intersection(utm_hot.geometry[j]).area < 1
                                    for i in range(len(hot)) for j in range(i + 1, len(hot))))
check("hotspot area = cells x cell area", np.allclose(utm_hot.geometry.area / 1e6, hot["area_km2"]))
check("camera inside hotspot 1 counted", hot.loc[0, "cameras"] == 1 and hot.loc[0, "camera median rate"] == 5.0)

## 9. Camera traps (local CSV files)

In [11]:
cam = bd.load_camera_sites(config)
check("222 camera sites after the 30 trap-day minimum", len(cam) == 222, f"{len(cam)}")
check("all cameras ran >= min_trap_days", (cam["trap_days"] >= config.min_trap_days).all())
check("camera rates are >= 0", (cam["rate"] >= 0).all())
check("cameras inside the box", cam["decimalLongitude"].between(west, east).all() and cam["decimalLatitude"].between(south, north).all())

buffalo_rate = bd.camera_species_rate(["buffalo"], config)
check("per-species camera rate matches load_camera_sites for buffalo",
      np.allclose(buffalo_rate.loc[cam["SiteID"]].values, cam["rate"].values))
lion = bd.camera_species_rate(["lionFemale", "lionMale"], config)
lioness = bd.camera_species_rate(["lionFemale"], config)
check("lion rate (both labels) >= female-only rate", (lion >= lioness - 1e-12).all())
days = bd.camera_detection_days(config).reindex(cam["SiteID"]).fillna(0).values
check("detection days <= trap-days", (days <= cam["trap_days"].values).all())

## 10. Cached real data

In [12]:
def cache(name):
    path = Path(config.cache_dir) / f"buffalo_{name}_{config.key()}.parquet"
    return pd.read_parquet(path) if path.exists() else None


presences = cache("presences")
if presences is None:
    skip("cached presences", "no cache file")
else:
    check("presences inside the box", presences["decimalLongitude"].between(west, east).all() and presences["decimalLatitude"].between(south, north).all())
    check("presences inside the years", presences["year"].between(config.start_year, config.end_year).all())
    check("658 presences after thinning", len(bd.thin_presences(presences, config)) == 658)

points = cache("points_embedded")
if points is None:
    skip("cached embeddings", "no cache file")
else:
    check("no missing embeddings", points[BAND_COLS].notna().all().all())
    check("embedding values within [-1, 1]", points[BAND_COLS].abs().max().max() <= 1.0001, f"max {points[BAND_COLS].abs().max().max():.3f}")
    counts = points.groupby(["year", "presence"]).size().unstack()
    check("10 background per presence every year", (counts[0] == 10 * counts[1]).all())
    pts = bv.add_utm(points, config.cell_size_m)
    closest = min(np.hypot(g.loc[g["presence"] == 1, "x"].values[:, None] - g.loc[g["presence"] == 0, "x"].values,
                           g.loc[g["presence"] == 1, "y"].values[:, None] - g.loc[g["presence"] == 0, "y"].values).min()
                  for _, g in pts.groupby("year"))
    check("real background >= buffer from same-year presences", closest >= config.buffer_m, f"closest {closest:.0f} m")

covariates = cache("serengeti_covariates")
if covariates is None:
    skip("cached covariates", "no cache file")
else:
    check("no missing covariates", covariates[COVARIATE_COLS].notna().all().all())
    check("NDVI in [-0.2, 1]", covariates[["ndvi_dry", "ndvi_wet"]].stack().between(-0.2, 1).all())
    check("tree cover in [0, 100] %", covariates["tree_cover_pct"].between(0, 100).all())
    check("elevation in [800, 2600] m", covariates["elevation_m"].between(800, 2600).all(),
          f"{covariates['elevation_m'].min():.0f}-{covariates['elevation_m'].max():.0f} m")
    check("slope in [0, 60] degrees", covariates["slope_deg"].between(0, 60).all())
    check("river distance in [0, 50 km]", covariates["dist_river_m"].between(0, bd.RIVER_SEARCH_M).all())

records = cache("mammal_records")
if records is None:
    skip("cached mammal records", "no cache file")
else:
    check("mammal records inside the box", records["decimalLongitude"].between(west, east).all() and records["decimalLatitude"].between(south, north).all())
    check("mammal records inside the years", records["year"].between(config.start_year, config.end_year).all())
    present = {name: int((records["speciesKey"] == sp["speciesKey"]).sum()) for name, sp in bv.SAFARI_SPECIES.items()}
    check("every safari species has records", all(v > 0 for v in present.values()), str(present))

grid = cache("serengeti_grid_2025_2500")
if grid is None:
    skip("cached map grid", "no cache file")
else:
    check("map grid inside the box", grid["decimalLongitude"].between(west, east).all() and grid["decimalLatitude"].between(south, north).all())
    offsets = ((grid["x"] - 1_250) % 2_500).round(3)
    check("map grid points sit on the 2.5 km lattice", ((offsets < 0.01) | (offsets > 2_499.99)).all())

## 11. Sightings heat map

In [13]:
if records is None:
    skip("heat map payload", "no cached mammal records")
else:
    species = {name: sp["speciesKey"] for name, sp in bv.SAFARI_SPECIES.items()}
    heat = sm.sightings_heat_map(records, species, config.bounds)
    layer = next(c for c in heat._children.values() if isinstance(c, sm.SightingsHeat))
    import json
    data = json.loads(layer.data)
    with_coords = records.dropna(subset=["decimalLongitude", "decimalLatitude"])
    check("every record with coordinates is in the map", len(data["x"]) == len(with_coords))
    for i, (name, key) in enumerate(species.items()):
        share = (with_coords["speciesKey"] == key).mean()
        check(f"{name}: average share matches the records", abs(data["p0"][i] - share) < 1e-12, f"{share:.3%}")
        check(f"{name}: colour-scale top above the average", share < data["vmax"][i] <= 1, f"{data['vmax'][i]:.3%}")
        check(f"{name}: species codes count its records", data["s"].count(i) == (with_coords["speciesKey"] == key).sum())

## Summary

In [14]:
summary = pd.DataFrame(results)
print(summary["result"].value_counts().to_string())
failed = summary[summary["result"] == "FAIL"]
assert failed.empty, f"{len(failed)} test(s) failed:\n{failed.to_string()}"
summary

result
PASS    77


,test,result,detail
0,UTM round trip,PASS,
1,UTM distance matches geodesic within 0.1 %,PASS,15685.2 m vs 15683.0 m
2,config key ignores cache_dir,PASS,
3,config key changes with buffer,PASS,
4,thinning keeps one per 1 km cell per year,PASS,"['a', 'c', 'd']"
...,...,...,...
72,cheetah: colour-scale top above the average,PASS,15.472%
73,cheetah: species codes count its records,PASS,
74,leopard: average share matches the records,PASS,2.589%
75,leopard: colour-scale top above the average,PASS,9.014%
